# Part 1a: Getting started with Keras

In this notebook we will train a small neural network on the LHC jet tagging dataset using Keras v3. When you are done, head straight to **`1c_hls4ml_synth.ipynb`** to convert the trained model to an FPGA design with hls4ml.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

import numpy as np
import sys
sys.path.append('..')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

%matplotlib inline
np.random.seed(0)

## Fetch the jet tagging dataset from Open ML

The [HLS4ML LHC jet dataset](https://openml.org/search?type=data&id=42468) was introduced by [Duarte et al. (2018)](https://arxiv.org/abs/1804.06913) to benchmark fast neural network inference on FPGAs for particle physics applications.

Jets are collimated sprays of particles produced when quarks or gluons are knocked out of colliding protons at the LHC. Identifying the origin of a jet in real time is a core task for LHC trigger systems, which must decide within a few microseconds whether to keep or discard each collision event.

The dataset contains 16 high-level jet substructure observables derived from simulated proton-proton collisions at √s = 13 TeV. These include energy correlation functions, N-subjettiness ratios, a groomed jet mass, and constituent multiplicity. The goal is to classify each jet into one of five categories:

| Label | Jet origin |
|-------|------------|
| `g`   | Gluon |
| `q`   | Light quark |
| `w`   | W boson decay (W → qq') |
| `z`   | Z boson decay (Z → qq') |
| `t`   | Top quark decay (t → bqq') |

In [ ]:
data = fetch_openml('hls4ml_lhc_jets_hlf')
X, y = data['data'], data['target']

### Let's print some information about the dataset


In [ ]:
print(data['feature_names'])
print(X.shape, y.shape)
print(X[:5])
print(y[:5])

As you saw above, the `y` target is an array of strings, e.g. `['g', 'w', ...]` etc.
We need to make this a "One Hot" encoding for the training.
Then, split the dataset into training and validation sets:

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y = np.eye(5)[y_encoded]  # one-hot encode
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(y[:5])

In [ ]:
scaler = StandardScaler()
X_train_val = scaler.fit_transform(X_train_val)
X_test = scaler.transform(X_test)

os.makedirs('../data/jet-tagging', exist_ok=True)
np.save('../data/jet-tagging/X_train_val.npy', X_train_val)
np.save('../data/jet-tagging/X_test.npy', X_test)
np.save('../data/jet-tagging/y_train_val.npy', y_train_val)
np.save('../data/jet-tagging/y_test.npy', y_test)
np.save('../data/jet-tagging/classes.npy', le.classes_)

## Now construct a model
We'll use 3 hidden layers with 64, then 32, then 32 neurons. Each layer will use ReLU activation.
Finally, we add an output layer with 5 neurons and the Softmax activation, to calculate the probability of each of the five classes.

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam

model = Sequential()
model.add(Dense(64, input_shape=(16,), name='fc1', activation='relu'))
model.add(Dense(32, name='fc2', activation='relu'))
model.add(Dense(32, name='fc3', activation='relu'))
model.add(Dense(5, name='output', activation='softmax'))
model.summary()

## Train the model
We'll use the Adam optimiser with categorical crossentropy loss.
The model isn't very complex, so this should take just a few minutes even on a CPU.

In [ ]:
model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(
    X_train_val,
    y_train_val,
    batch_size=1024,
    epochs=20,
    validation_split=0.25,
    shuffle=True,
)
os.makedirs('../models', exist_ok=True)
model.save('../models/keras_model_part1.h5')

## Check performance
Check the accuracy and make a ROC curve:

In [ ]:
import plotting
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

y_keras = model.predict(X_test)
print("Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
plt.figure(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_keras, le.classes_)

An accuracy of ~75% is expected for this 5-class problem — random guessing gives only 20%, and some classes (notably gluon vs. light quark) are physically very similar and genuinely hard to separate even with more sophisticated methods.

The ROC (Receiver Operating Characteristic) curve shows, for each class, the trade-off between signal efficiency (true positive rate) and background efficiency (false positive rate) as the decision threshold is varied. The area under the curve (AUC) ranges from 0.5 (random classifier) to 1.0 (perfect). Higher and further to the upper-left is better.

**N.B.** This notebook trains a full-precision (32-bit floating-point) model. When converting to an FPGA design, hls4ml applies post-training quantization (PTQ) by default, which works well at 16-bit precision but struggles to match accuracy below ~8 bits. For the most resource-efficient FPGA designs **quantization-aware training (QAT)** gives substantially better results. See **Part 2** for QKeras (Keras) and Brevitas (PyTorch) QAT examples.

## Next step

Your model is trained and saved. Open **`1c_hls4ml_synth.ipynb`** to convert it to an FPGA design with hls4ml.